# Курсовая работа: Численное моделирование, феномен Рунге на python

## 1. Введение и постановка задачи
В данной работе исследуется задача приближения функций с помощью полиномиальной интерполяции. Особое внимание уделяется классической проблеме вычислительной математики — **феномену Рунге**, который заключается в резком возрастании ошибки интерполяции на краях отрезка при использовании равномерной сетки узлов.

В качестве эталонной функции рассматривается функция Рунге:
$$f(x) = \frac{1}{1 + 25x^2}$$
на отрезке $[-1, 1]$.

## 2. Математическая модель
### 2.1. Интерполяционный многочлен Лагранжа
Для заданного набора из $n+1$ узлов $(x_0, x_1, \dots, x_n)$ и значений функции в них $(y_0, y_1, \dots, y_n)$, строится многочлен Лагранжа:
$$L_n(x) = \sum_{i=0}^{n} y_i \cdot l_i(x)$$
где базисные полиномы $l_i(x)$ вычисляются как:
$$l_i(x) = \prod_{j=0, j \neq i}^{n} \frac{x - x_j}{x_i - x_j}$$

### 2.2. Типы сеток и узлы Чебышева
Для сравнения в работе используются два типа сеток:
1. **Равномерная сетка:** Узлы распределены на равном расстоянии друг от друга.
2. **Сетка Чебышева (оптимальная):** Для минимизации максимальной погрешности используются корни многочленов Чебышева первого рода:
$$x_k = \cos\left(\frac{2k - 1}{2(n+1)}\pi\right)$$



## 3. Программная реализация и анализ графиков

Для визуализации процесса используется библиотека `matplotlib`, а для создания интерактивного окна — `tkinter`. Оценка максимальной погрешности между точным значением функции и интерполяционным полиномом производится с помощью векторизованных вычислений в библиотеке `pandas`.

Интерфейс программы позволяет исследователю:
* Задавать любую степень полинома $n$.
* Строить графики для равномерной сетки, чтобы зафиксировать появление осцилляций (феномен Рунге).
* Строить графики по узлам Чебышева для демонстрации сходимости полинома к эталонной функции.

In [ ]:
import tkinter as tk
from tkinter import messagebox
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# ==========================================
# 1. МАТЕМАТИЧЕСКАЯ ЧАСТЬ
# ==========================================

# Эталонная функция Рунге
def runge_function(x):
    return 1 / (1 + 25 * x**2)

# Вычисление полинома Лагранжа
def calc_lagrange(x_dense, x_nodes, y_nodes):
    y_dense = np.zeros_like(x_dense, dtype=float)
    n = len(x_nodes)
    
    # Проходим по всем узлам и строим базисные полиномы
    for i in range(n):
        p = np.ones_like(x_dense, dtype=float)
        for j in range(n):
            if i != j:
                p *= (x_dense - x_nodes[j]) / (x_nodes[i] - x_nodes[j])
        y_dense += y_nodes[i] * p
        
    return y_dense

# ==========================================
# 2. ЛОГИКА ПОСТРОЕНИЯ И АНАЛИЗА
# ==========================================

def plot_graph(grid_type):
    try:
        n = int(entry_n.get()) # Получаем степень из текстового поля
    except ValueError:
        messagebox.showerror("Ошибка", "Введите целое число!")
        return

    # Количество узлов на 1 больше, чем степень полинома
    num_nodes = n + 1 
    
    # Генерация узлов X в зависимости от нажатой кнопки
    if grid_type == "uniform":
        x_nodes = np.linspace(-1, 1, num_nodes)
        title = f"Равномерная сетка (n={n})"
    else:
        # Узлы Чебышева
        k = np.arange(1, num_nodes + 1)
        x_nodes = np.cos((2 * k - 1) / (2 * num_nodes) * np.pi)
        title = f"Узлы Чебышева (n={n})"
        
    y_nodes = runge_function(x_nodes)

    # Создаем густую сетку для плавной отрисовки графика (500 точек)
    x_dense = np.linspace(-1, 1, 500)
    y_exact = runge_function(x_dense)
    y_interp = calc_lagrange(x_dense, x_nodes, y_nodes)

    # ИСПОЛЬЗОВАНИЕ PANDAS ДЛЯ АНАЛИЗА ОШИБКИ
    # Собираем данные в таблицу DataFrame для удобного подсчета
    df = pd.DataFrame({
        "Точное значение": y_exact,
        "Интерполяция": y_interp
    })
    # Вычисляем колонку с ошибками и находим максимум
    df["Ошибка"] = np.abs(df["Точное значение"] - df["Интерполяция"])
    max_error = df["Ошибка"].max()
    
    # Обновляем текст с ошибкой в интерфейсе
    label_error.config(text=f"Максимальная погрешность: {max_error:.5f}")

    # Очищаем старый график и рисуем новый
    ax.clear()
    ax.plot(x_dense, y_exact, 'b-', label='Функция Рунге', linewidth=2)
    ax.plot(x_dense, y_interp, 'r--', label='Полином Лагранжа')
    ax.plot(x_nodes, y_nodes, 'ko', label='Узлы интерполяции') # Черные точки
    
    ax.set_title(title)
    ax.grid(True)
    ax.legend()
    # Фиксируем масштаб по оси Y, чтобы видеть суть феномена Рунге
    ax.set_ylim([-0.5, 1.5]) 
    
    # Перерисовываем холст
    canvas.draw()

# Обертки для кнопок
def btn_uniform_click():
    plot_graph("uniform")

def btn_chebyshev_click():
    plot_graph("chebyshev")

# ==========================================
# 3. ГРАФИЧЕСКИЙ ИНТЕРФЕЙС 
# ==========================================

root = tk.Tk()
root.title("Моделирование феномена Рунге")

# Верхняя панель с кнопками
frame_top = tk.Frame(root, pady=10)
frame_top.pack()

tk.Label(frame_top, text="Степень полинома (n):").pack(side=tk.LEFT)
entry_n = tk.Entry(frame_top, width=5)
entry_n.insert(0, "10") # Значение по умолчанию
entry_n.pack(side=tk.LEFT, padx=10)

btn_uniform = tk.Button(frame_top, text="Построить (Равномерная сетка)", command=btn_uniform_click)
btn_uniform.pack(side=tk.LEFT, padx=5)

btn_chebyshev = tk.Button(frame_top, text="Построить (Сетка Чебышева)", command=btn_chebyshev_click)
btn_chebyshev.pack(side=tk.LEFT, padx=5)

# Метка для вывода ошибки
label_error = tk.Label(root, text="Максимальная погрешность: -", fg="red", font=("Arial", 12, "bold"))
label_error.pack(pady=5)

# Область для графика Matplotlib
fig, ax = plt.subplots(figsize=(8, 5))
canvas = FigureCanvasTkAgg(fig, master=root)
canvas.get_tk_widget().pack()

# Запуск программы
root.mainloop()